In [95]:
import yfinance as yf
import numpy as np
import pandas as pd
import pandas_datareader as web
import matplotlib.pyplot as plt
import statsmodels.api as sm
import datetime

In [96]:
def get_date(start_date, end_date = None):
    '''
    fetches dates for ff3 and stock data.
    deploy this function to reduce complexity/potential misalignment of data
    Formatting for dates: "YYYY-MM-DD"
    '''
    if end_date == None:
        end_date = datetime.datetime.now()
    else:
        end_date = pd.to_datetime(end_date)

    start_date = pd.to_datetime(start_date)

    return start_date, end_date

In [97]:
def fetch_ff3_data(start_date, end_date = None):
    '''
    Fetching FF3 Factor data for a given timeframe using Ken French API
    '''
    start_date, end_date = get_date(start_date, end_date)
    ff_data = web.DataReader('F-F_Research_Data_Factors', 'famafrench', start=start_date, end=end_date)
    del(ff_data["DESCR"], ff_data[1])
    ff_data = pd.DataFrame(ff_data[0]) / 100
    return(ff_data)

In [98]:
# adjusted version:
def get_portfolio(ticker_list, start_date, end_date = None):
    '''
    Creates a Dataframe that contains monthly returns for all portfolios
    Input data should be a dict of the following form:

    {
    "Portfolio i" : [Tickers],
    "Portfolio i+1" : [Tickers]
    }
    '''
    start_date, end_date = get_date(start_date, end_date)

    portfolio_dict = {}
    
    for x in ticker_list.keys():
        
        daily = yf.download(ticker_list[x], start=start_date, end=end_date)

        close = daily["Close"]
        
        monthly_close = close.resample("ME").agg("last").to_period("M")
    
        monthly_returns = monthly_close.pct_change().dropna()

        portfolio_dict[x] = monthly_returns.mean(axis = 1)
        
    return pd.DataFrame(portfolio_dict)

# I dislike the way ticker_list is set up; there should be a more efficient and intuitive way to solve this and download the portfolios
# get rid of the for loop with yf.download, it calls API every time and it is generally slow, instead use yf.download to your advantage

In [339]:
# New for timeseries
def timeseries_prep(portfolio_df, ff3_data):
    '''
    Creating a function that cleans and prepares our stock data for regression
    '''
    joined_df = portfolio_df.join(ff3_data, how="inner")

    portfolio_cols = portfolio_df.columns
    ff3_cols = ff3_data.columns[:-1]
    
    joined_df[portfolio_cols] = joined_df[portfolio_cols].sub(joined_df["RF"],axis = 0)
    
    portfolio_excess_returns = joined_df[portfolio_cols]
    risk_factors = sm.add_constant(joined_df[ff3_cols])
    return portfolio_excess_returns, risk_factors

In [334]:
def timeseries(portfolio_excess_returns, risk_factors):
    '''
    time series regression to compute the factor loadings
    '''
    model = sm.OLS(portfolio_excess_returns, risk_factors)
    results = model.fit()

    factor_loadings_t = results.params
    factor_loadings_t.columns = Y.columns

    factor_loadings = factor_loadings_t.T
    factor_loadings.columns.values[0] = "jensen_alpha"
    
    return factor_loadings

In [336]:
def crosssection(portfolio_excess_returns, factor_loadings):
    '''
    prepare the betas for regression; Since OLS takes the Form BETA = (X'X)^-1 X' and betas stay constant in this regression,
    we will use this attribute to compute the regression instead.
    '''
    factor_loadings = factor_loadings.drop("jensen_alpha",axis = 1)
    betas = sm.add_constant(factor_loadings)
    
    cols = betas.columns
    projection_matrix = np.linalg.inv(betas.T @ betas) @ betas.T
    gammas = projection_matrix @ Y.T

    gammas.index = cols
    gammas = gammas.T
    return gammas
    

In [104]:
### Test Block

In [105]:
my_portfolios = {
    "Tech_Mix": ["AAPL", "MSFT", "NVDA"],
    "Retail_Mix": ["WMT", "TGT", "COST"],
    "Single_Stock": ["TSLA"]
}


ff3 = fetch_ff3_data("2020-12-31")
new_pf = get_portfolio(my_portfolios, "2020-12-31")

print(new_pf.head)

/var/folders/lg/8378lyw55rqdlc0qt63jgcy00000gn/T/ipykernel_54416/2951705467.py:6: FutureWarning: The argument 'date_parser' is deprecated and will be removed in a future version. Please use 'date_format' instead, or read your data in as 'object' dtype and then call 'to_datetime'.
  ff_data = web.DataReader('F-F_Research_Data_Factors', 'famafrench', start=start_date, end=end_date)
/var/folders/lg/8378lyw55rqdlc0qt63jgcy00000gn/T/ipykernel_54416/2951705467.py:6: FutureWarning: The argument 'date_parser' is deprecated and will be removed in a future version. Please use 'date_format' instead, or read your data in as 'object' dtype and then call 'to_datetime'.
  ff_data = web.DataReader('F-F_Research_Data_Factors', 'famafrench', start=start_date, end=end_date)
/var/folders/lg/8378lyw55rqdlc0qt63jgcy00000gn/T/ipykernel_54416/1497456591.py:18: FutureWarning: YF.download() has changed argument auto_adjust default to True
  daily = yf.download(ticker_list[x], start=start_date, end=end_date)
[**

<bound method NDFrame.head of          Tech_Mix  Retail_Mix  Single_Stock
Date                                       
2021-01  0.010797   -0.021244      0.124506
2021-02 -0.006600   -0.039350     -0.148740
2021-03 -0.001481    0.064834     -0.011207
2021-04  0.090091    0.044777      0.062147
2021-05  0.008052    0.044720     -0.118713
...           ...         ...           ...
2025-09  0.063452   -0.007163      0.332015
2025-10  0.048939    0.000500      0.026623
2025-11 -0.047188    0.028077     -0.057802
2025-12  0.003870    0.010927      0.045447
2026-01 -0.029156    0.105506     -0.018300

[61 rows x 3 columns]>


In [340]:
portfolio_excess_returns, risk_factors = timeseries_prep(new_pf, ff3)
factor_loadings = timeseries(portfolio_excess_returns, risk_factors)
gammas = crosssection(portfolio_excess_returns, factor_loadings)


print(f"\n {gammas.head}")


 <bound method NDFrame.head of             const    Mkt-RF       SMB       HML
Date                                           
2021-01 -0.044842 -0.013776  0.030011 -0.040820
2021-02 -0.038739  0.029738 -0.065584 -0.046115
2021-03  0.089419 -0.008319  0.021749  0.089681
2021-04 -0.013212  0.035529 -0.031001 -0.039095
2021-05  0.061584  0.020486 -0.033810  0.051798
2021-06 -0.065907  0.061256 -0.063901 -0.106493
2021-07  0.056181  0.007371  0.002733  0.047111
2021-08 -0.046130  0.034399 -0.033555 -0.069679
2021-09 -0.015244 -0.044492  0.056807  0.011019
2021-10 -0.012233 -0.003358  0.078279 -0.036449
2021-11 -0.108940  0.071724 -0.092978 -0.149772
2021-12  0.022490  0.011043 -0.022221  0.018715
2022-01 -0.012357 -0.020736  0.001309  0.008863
2022-02 -0.013054 -0.001124 -0.013246 -0.007080
2022-03  0.056165 -0.016750  0.069498  0.048495
2022-04  0.158934 -0.068369  0.068480  0.204598
2022-05 -0.231012  0.038922 -0.103459 -0.233045
2022-06  0.021058 -0.036210  0.024205  0.050004
2022-07 

In [47]:
my_portfolios = {
    "Tech_Mix": ["AAPL", "MSFT", "NVDA"],
    "Retail_Mix": ["WMT", "TGT", "COST"],
    "Single_Stock": ["TSLA"]
}


ff3 = fetch_ff3_data("2020-12-31")
new_pf = get_portfolio(my_portfolios, "2020-12-31")

Y, X = reg_prep(new_pf, ff3)

smry = regress(Y, X)

for i in smry.keys():
    print(smry[i].resid)

/var/folders/lg/8378lyw55rqdlc0qt63jgcy00000gn/T/ipykernel_54416/2951705467.py:6: FutureWarning: The argument 'date_parser' is deprecated and will be removed in a future version. Please use 'date_format' instead, or read your data in as 'object' dtype and then call 'to_datetime'.
  ff_data = web.DataReader('F-F_Research_Data_Factors', 'famafrench', start=start_date, end=end_date)
/var/folders/lg/8378lyw55rqdlc0qt63jgcy00000gn/T/ipykernel_54416/2951705467.py:6: FutureWarning: The argument 'date_parser' is deprecated and will be removed in a future version. Please use 'date_format' instead, or read your data in as 'object' dtype and then call 'to_datetime'.
  ff_data = web.DataReader('F-F_Research_Data_Factors', 'famafrench', start=start_date, end=end_date)
/var/folders/lg/8378lyw55rqdlc0qt63jgcy00000gn/T/ipykernel_54416/1497456591.py:18: FutureWarning: YF.download() has changed argument auto_adjust default to True
  daily = yf.download(ticker_list[x], start=start_date, end=end_date)
[**

Date
2021-01    0.037846
2021-02   -0.005896
2021-03   -0.017119
2021-04   -0.012464
2021-05    0.035991
2021-06    0.032039
2021-07   -0.029640
2021-08    0.023312
2021-09    0.012308
2021-10    0.036710
2021-11    0.125284
2021-12   -0.046645
2022-01    0.056209
2022-02    0.010355
2022-03   -0.008676
2022-04   -0.019366
2022-05    0.019801
2022-06   -0.046673
2022-07   -0.012302
2022-08   -0.050725
2022-09   -0.033846
2022-10    0.000944
2022-11    0.023198
2022-12   -0.031262
2023-01    0.034052
2023-02    0.085575
2023-03    0.021322
2023-04   -0.008909
2023-05    0.083536
2023-06   -0.024448
2023-07    0.003002
2023-08   -0.009336
2023-09   -0.027231
2023-10    0.013668
2023-11   -0.005463
2023-12   -0.014316
2024-01    0.021934
2024-02   -0.013874
2024-03   -0.002716
2024-04   -0.009478
2024-05    0.066132
2024-06    0.006735
2024-07    0.000031
2024-08   -0.045252
2024-09   -0.043275
2024-10   -0.001876
2024-11   -0.051722
2024-12    0.000873
2025-01   -0.113643
2025-02    0.03

In [ ]:
# testing an idea for cross section, Y = row_i of portfolio matrix, essentially we can just transpose it and use it as new Y
# X = model.params
# should we add the gamma average here alr? (new model.params.mean())

def cs_reg(Y, beta):
    '''
    Performs cross sectional regression.
    Intended to be used to regress Y = Stock returns for month i 
    X 
    '''
    cross_section = Y.transpose()
    X = sm.add_constant(beta)
    for i in Y.columns:
        model = sm.OLS(Y[i], X)
        results = model.fit()
        regression_data[i] = results
    return regression_data

    # results.params.mean()

    return results

In [8]:
def print_results(regression_results):
    '''
    prints summaries for the regressions of every singular stock
    '''
    print([regression_results[i].summary() for i in regression_results])

In [9]:
# iterate through the results, create key value pairs for every stock with params, p values, tstat
# 1. need dict with models
# 2. create new empty dictionary
# 3. write loop. For every key, we create a new key in the dict that holds params, pvalues, tstat and bse
# 4. return dictionary
# maybe create a dictionary, where index is params, pvalues etc. and columns are stocks

# goal is: pvalues so we can filter using masks, only the significant ones
# ttest requires mean of significant coefficients

reg_data = regress(Y=Y, X=X)

parameters = {}
pvalues = {}
tstat = {}
standard_error = {}


for i in reg_data.keys():
    parameters[i] = reg_data[i].params
    pvalues[i] = reg_data[i].pvalues
    tstat[i] = reg_data[i].tvalues
    standard_error[i] = reg_data[i].bse

print(parameters)

#for i in reg_data.keys():
    #print(reg_data[i].pvalues)

NameError: name 'Y' is not defined

In [10]:
def summarize(regression_results):
    '''
    retrieves dataframe of p-values, coefficients, std err
    '''
    
    

In [11]:
def stock_plot(Y, title="Stock Alpha", xlabel="Date", ylabel="Returns", grid=True):
    '''
    plots any dataframe of stocks
    x-axis default: date (PeriodIndex: "YYYY-MM"
    y-axis default: Stock Alpha (in percent)
    '''

    if not isinstance(title, str):
        raise TypeError(f"Expected String input, instead received {type(title)}")
    if not isinstance(xlabel, str):
        raise TypeError(f"Expected String input, instead received {type(xlabel)}")
    if not isinstance(ylabel, str):
        raise TypeError(f"Expected String input, instead received {type(ylabel)}")
    if not isinstance(grid, bool):
        raise TypeError(f"Expected True or False, instead received {type(grid)}")
    
    plt.figure(figsize=(10,6))
    Y.plot()
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.grid(grid)
    plt.show

In [12]:
# TESTING BLOCK:

In [13]:
### Testing Block for get_portfolio
my_portfolios = {
    "Tech_Mix": ["AAPL", "MSFT", "NVDA"],
    "Retail_Mix": ["WMT", "TGT", "COST"],
    "Single_Stock": ["TSLA"]
}


ff3 = fetch_ff3_data("2020-12-31")
new_pf = get_portfolio(my_portfolios, "2020-12-31")

print(new_pf.head())
Y, X, joined_df= reg_prep(new_pf, ff3)

print(joined_df["RF"])

/var/folders/lg/8378lyw55rqdlc0qt63jgcy00000gn/T/ipykernel_4460/2329612805.py:6: FutureWarning: The argument 'date_parser' is deprecated and will be removed in a future version. Please use 'date_format' instead, or read your data in as 'object' dtype and then call 'to_datetime'.
  ff_data = web.DataReader('F-F_Research_Data_Factors', 'famafrench', start=start_date, end=end_date)
/var/folders/lg/8378lyw55rqdlc0qt63jgcy00000gn/T/ipykernel_4460/2329612805.py:6: FutureWarning: The argument 'date_parser' is deprecated and will be removed in a future version. Please use 'date_format' instead, or read your data in as 'object' dtype and then call 'to_datetime'.
  ff_data = web.DataReader('F-F_Research_Data_Factors', 'famafrench', start=start_date, end=end_date)
/var/folders/lg/8378lyw55rqdlc0qt63jgcy00000gn/T/ipykernel_4460/54992587.py:18: FutureWarning: YF.download() has changed argument auto_adjust default to True
  daily = yf.download(ticker_list[x], start=start_date, end=end_date)
[*******

         Tech_Mix  Retail_Mix  Single_Stock
Date                                       
2021-01  0.010797   -0.021244      0.124506
2021-02 -0.006600   -0.039350     -0.148740
2021-03 -0.001481    0.064834     -0.011207
2021-04  0.090091    0.044777      0.062147
2021-05  0.008052    0.044720     -0.118713



/var/folders/lg/8378lyw55rqdlc0qt63jgcy00000gn/T/ipykernel_4460/54992587.py:22: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  monthly_close = close.resample("M").agg("last").to_period("M")


ValueError: not enough values to unpack (expected 3, got 2)

In [ ]:
stock_plot(Y)

In [ ]:
# 1. get_date, 2. fetch_ff3_data, 3. fetch_stock, 4. prep_stocks, 5. reg_prep, 6. regress
tickers = ["AAPL", "MSFT", "GOOGL", "AMZN", "TSLA"]

start, end = get_date("2020-12-31")
ff3 = fetch_ff3_data(start, end)
stock_data = prep_stocks(tickers, start, end) # if fetch stocks, theres an error I think if used a list
X, Y = reg_prep(stock_data, ff3)
regress(X=X, Y=Y)
stock_plot(Y)

In [ ]:
regress(Y,X)

In [ ]:
tickers = ["AAPL", "MSFT", "GOOGL", "AMZN", "TSLA"]

start, end = get_date("2020-12-31")
ff3 = fetch_ff3_data(start, end)
stocks_df = prep_stocks(tickers, start, end)
X, Y = reg_prep(stocks_df, ff3)

print(f" {X} \n\n {Y}")

In [ ]:
B = fetch_stock("AAPL", start, end)

stock_plot(B)
stock_plot(Y)

In [ ]:
def get_date(start_date, end_date=None):
    '''
    fetches dates for ff3 and stock data.
    deploy this function to reduce complexity/potential misalignment of data
    Formatting for dates: "YYYY-MM-DD"
    '''
    if end_date == None:
        end_date = datetime.datetime.now()
    else:
        pd.to_datetime(end_date)

    start_date = pd.to_datetime(start_date)

    return start_date, end_date

In [ ]:
def fetch_stock(stock_ticker, start_date, end_date):
    '''
    fetching daily stock returns, which are then converted to monthly stock data in order to match ff3 data
    '''

    resample_logic ={"Close":"last"}

    daily = yf.Ticker(stock_ticker).history(start=start_date, end=end_date)
        
    monthly = daily.resample("M").agg(resample_logic).to_period("M")
    
    monthly_returns = monthly.pct_change()
    return monthly_returns

In [ ]:
# Function for plotting Data

def stock_plot(stock_arr, plt_title="", labelx="Date (YYYY-MM)", labely="", grid_stat=True):

    if not isinstance(plt_title, str):
        raise TypeError(f"Expected String input, instead received {type(title)}")
    if not isinstance(labelx, str):
        raise TypeError(f"Expected String input, instead received {type(labelx)}")
    if not isinstance(labely, str):
        raise TypeError(f"Expected String input, instead received {type(labely)}")
    if not isinstance(grid_stat, bool):
        raise TypeError(f"Expected True or False, instead received {type(grid_stat)}")

    # x_axis = np.linspace(start=arr.index.year[1], stop=arr.index.year[-1], num= arr.index.year.size)
    
    plt.figure(figsize=(10, 6))
    arr.plot()
    plt.title(plt_title)
    plt.ylabel(labely)
    plt.xlabel(labelx)
    plt.grid(grid_stat)
    plt.show()



In [ ]:
def fetch_data(stock_ticker, start_date, end_date):

    #fetching stock data

    resample_logic ={"Close":"last"}

    daily = yf.Ticker(stock_ticker).history(start=start_date, end=end_date)
        
    monthly = daily.resample("M").agg(resample_logic).to_period("M")
    
    monthly_returns = monthly.pct_change()

    #fetching ff3 data for according period

    ff_data = web.DataReader('F-F_Research_Data_Factors', 'famafrench', start=start_date, end=end_date)
    del(ff_data["DESCR"], ff_data[1])
    ff_data = pd.DataFrame(ff_data[0]) / 100

    #combine both dataframes, matching indices

    combined_df = monthly_returns.join(ff_data, how="inner").dropna()
    return(combined_df)

In [ ]:
ff3 = fetch_ff3_data("2018-01-01", "2024-12-31")
apple = fetch_data("AAPL", "2017-12-31", "2024-12-31")

apple

In [ ]:
Y, X = prep_data(apple, ff3)


In [ ]:
reg_df = pd.concat([apple, ff3], axis = 1)
print(reg_df)

In [ ]:
    reg_df = pd.merge(stock_arr, ff3_data, left_index=True, right_index=True, how='inner')

    reg_df["Excess_Returns"] = reg_df["Relative Returns"] - reg_df["RF"]

    Y = reg_df["Excess_Returns"]

    X = reg_df[['Mkt-RF', 'SMB', 'HML']]

In [ ]:
stock_plot(ff3, "Excess_Returns", "Date", "Excess Returns", True)

In [ ]:
stock_plot(arr=apple_er, plt_title="Excess Returns", labely="Returns (in pct)")

In [ ]:
print(OLS_Regression(stock_data, regressors))